# 보행 신호등 검출기 학습 (Colab)

YOLO11n · 3클래스(`ped_light` / `car_light` / `traffic_sign`) · AI Hub 056 데이터.

**추론 시에는 `ped_light` 만 사용한다.** 나머지 둘은 명시적 hard negative 로,
생김새가 비슷한 차량 신호등과 "신호등 옆 빨간 간판"(프로토타입 실패 원인)을
모델이 직접 배우게 하는 역할이다.

### 로컬(GTX 1060 3GB)에서 확인한 것
| imgsz | batch | 전체/에폭 | peak VRAM |
|---|---|---|---|
| 640 | 8 | 10.8분 | 1.19GB |
| 1024 | 4 | 27.0분 | 1.48GB |
| 1024 | 6 | 67.8분 | 2.19GB (스래싱) |

Pascal 은 FP16 텐서 코어가 없어 AMP 가 메모리만 아낄 뿐 연산 가속이 없었다.
T4 는 텐서 코어가 있으므로 **`imgsz=1024` 로 되돌린다.** 보행등이
640 에서 6×12px, 1024 에서 10×20px 이라 이 차이가 검출률을 좌우한다.

### 순서
1. GPU 확인 → 2. 설치 → 3. 데이터 준비 → 4. 학습 → 5. 평가 → 6. 다운로드

## 1. GPU 확인

T4 이상이 아니면 런타임을 다시 할당받는 편이 낫다.

In [ ]:
!nvidia-smi

import torch
print()
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.1f}GB  sm_{p.major}{p.minor}")
    # 텐서 코어는 sm_70 이상. 이게 없으면 AMP 가속이 없어 로컬 1060 과 다를 바 없다
    if p.major >= 7:
        print("텐서 코어 있음 → AMP 가속 유효. imgsz=1024 로 진행")
    else:
        print("[경고] 텐서 코어 없음. 런타임 > 런타임 유형 변경 에서 GPU 재할당 권장")
else:
    print("[오류] GPU 미할당. 런타임 > 런타임 유형 변경 > 하드웨어 가속기 = GPU")


## 2. 설치

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. 데이터 준비

`dataset_det_1024.zip`(약 1GB)을 Google Drive 아무 곳에나 올려두고 아래 경로만 맞춘다.

**중요: 마운트된 Drive 에서 직접 학습하면 안 된다.** Drive 는 I/O 가 느려서
로컬 GPU 보다 느려질 수 있다. 반드시 `/content`(로컬 SSD)로 풀어서 쓴다.

### 압축 해제가 느린 이유와 대응
파일이 22,357개(jpg 11,177 + txt 11,177)다. 용량보다 **개수**가 비용이다.

- Drive FUSE 는 처리량이 10~30MB/s 라 1GB 읽기에만 1~4분
- 작은 파일 다수는 파일당 시스콜 비용이 지배적

그래서 두 단계로 나눈다. **Drive→로컬 복사는 순차 읽기**라 빠르고,
그다음 로컬끼리 푸는 것은 FUSE 를 타지 않는다. 해제도 Python `zipfile` 대신
시스템 `unzip`(C 구현, 버퍼링이 낫다)을 쓴다. 합쳐서 2~4분 정도.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ★ 업로드한 zip 경로를 여기서 맞춘다
ZIP = '/content/drive/MyDrive/dataset_det_1024.zip'


In [ ]:
import os, shutil, time
from pathlib import Path

DST   = Path('/content/dataset')
LOCAL = Path('/content/dataset_det_1024.zip')
assert Path(ZIP).exists(), f'zip 이 없다: {ZIP}'

# 1단계: Drive → 로컬 SSD 순차 복사 (FUSE 랜덤 접근을 피한다)
if not LOCAL.exists():
    t0 = time.time()
    shutil.copyfile(ZIP, LOCAL)
    mb = LOCAL.stat().st_size / 1e6
    dt = time.time() - t0
    print(f'복사 {mb/1000:.2f}GB  {dt:.0f}초  ({mb/max(dt,1):.0f} MB/s)')
else:
    print('이미 복사되어 있음')


In [ ]:
# 2단계: 로컬끼리 해제. 시스템 unzip 이 Python zipfile 보다 빠르다
import time
t0 = time.time()
!mkdir -p {DST}
!unzip -q -o {LOCAL} -d {DST}
print(f'해제 {time.time()-t0:.0f}초')


In [ ]:
for split in ('train', 'val'):
    n_img = len(list((DST/'images'/split).glob('*.jpg')))
    n_lab = len(list((DST/'labels'/split).glob('*.txt')))
    print(f'  {split}: 이미지 {n_img}, 라벨 {n_lab}')
assert len(list((DST/'images'/'train').glob('*.jpg'))) == 6230, '학습 이미지 수가 다르다'
print('데이터 준비 완료')


### data.yaml 재생성

zip 안의 yaml 과 목록 파일은 Windows 절대경로라 여기서 못 쓴다. Colab 경로로 다시 만든다.

학습 중 검증은 1,000장 서브셋으로 한다 (전체 4,947장을 매 에폭 돌리는 것은 낭비).
최종 평가만 전체로 한다.

In [ ]:
NAMES = ['ped_light', 'car_light', 'traffic_sign']

# 파일명만 담긴 목록 → Colab 절대경로로 복원
names = (DST/'val_1000_names.txt').read_text().split()
val_paths = [str(DST/'images'/'val'/n) for n in names]
val_paths = [p for p in val_paths if os.path.exists(p)]
(DST/'val_1000.txt').write_text('\n'.join(val_paths) + '\n')
print(f'검증 서브셋 {len(val_paths)}장')

def write_yaml(path, val_field):
    Path(path).write_text(
        f"path: {DST}\ntrain: images/train\nval: {val_field}\n"
        f"nc: {len(NAMES)}\nnames: {NAMES}\n")

write_yaml(DST/'data.yaml', 'val_1000.txt')       # 학습 중 검증
write_yaml(DST/'data_fullval.yaml', 'images/val') # 최종 평가
print((DST/'data.yaml').read_text())


## 4. 학습

**첫 에폭 시간이 찍히면 전체 소요를 바로 알 수 있다.** 로컬 1060 은 1024/b4 에서
27분/에폭이었다. T4 에서 5분 이내면 예상대로고, 15분을 넘으면 GPU 할당이
기대와 다르니 `IMGSZ=640` 으로 낮추는 편이 낫다.

- `rect=False` — `rect=True` 는 셔플을 끈다. 우리 데이터는 종횡비가 전부 같아
  정렬 이득이 없으므로 셔플을 지키는 쪽이 낫다.
- `cache=False` — RAM 캐시는 Colab 메모리를 넘길 수 있다. 로컬 SSD 가 충분히 빠르다.
- `patience` — 개선이 없으면 조기 종료해서 시간을 아낀다.

In [ ]:
IMGSZ   = 1024   # T4 기준. 느리면 640 으로
BATCH   = 16     # 16GB VRAM 기준. OOM 나면 8
EPOCHS  = 25


In [ ]:
import time
from ultralytics import YOLO

t0 = time.time()
ep = []

def on_epoch_end(trainer):
    ep.append(time.time() - t0 - sum(ep))
    n, avg = len(ep), sum(ep)/len(ep)
    print(f"\n[진행] {n}/{EPOCHS} 에폭  평균 {avg/60:.1f}분/에폭  "
          f"예상 잔여 {(EPOCHS-n)*avg/3600:.1f}시간\n", flush=True)

model = YOLO('yolo11n.pt')
model.add_callback('on_fit_epoch_end', on_epoch_end)

model.train(
    data=str(DST/'data.yaml'),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=2, amp=True, cache=False, rect=False,
    patience=15, project='/content/runs/det', name='ped_light',
    exist_ok=True, seed=0, plots=True, verbose=True,
)
print(f"\n총 {(time.time()-t0)/3600:.2f}시간")


## 5. 최종 평가 (전체 val 4,947장)

학습 중 검증은 1,000장 서브셋이므로 최종 수치는 여기서 다시 낸다.

**`ped_light` 의 mAP50 과 재현율(R)이 판단 기준이다.** 내일 통합했을 때
"검출이 안 되는 건지, 붙이는 게 잘못된 건지" 구분하려면 이 숫자가 필요하다.

In [ ]:
BEST = '/content/runs/det/ped_light/weights/best.pt'

m = YOLO(BEST)
metrics = m.val(data=str(DST/'data_fullval.yaml'), imgsz=IMGSZ, split='val')

print('\n클래스별 결과')
for i, name in enumerate(NAMES):
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f'  {name:14s} P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}  mAP50-95={ap:.3f}')


### 크기별 검출률 (선택)

보행등은 대부분 작다. 어느 크기부터 검출이 무너지는지 알면
배포 시 "몇 미터부터 안내 가능한가"를 정하는 근거가 된다.

In [ ]:
import numpy as np
from collections import Counter
from pathlib import Path

hs = []
for t in (DST/'labels'/'val').glob('*.txt'):
    for line in t.read_text().splitlines():
        f = line.split()
        if f and f[0] == '0':            # ped_light
            hs.append(float(f[4]) * 317)  # 정규화 h → 픽셀(원본 크롭 높이 317)
hs = np.array(hs)
print(f'val ped_light {len(hs)}개')
for lo, hi in ((0,10), (10,20), (20,40), (40,1000)):
    print(f'  높이 {lo:3d}~{hi:<4d}px : {((hs>=lo)&(hs<hi)).sum():5d}개')


## 6. 가중치 다운로드

`best.pt` 는 5MB 정도다. 받아서 저장소의 `models/` 에 두면 된다.

In [ ]:
from google.colab import files
import shutil

# Drive 에도 사본을 남긴다 (세션이 끊겨도 보존)
shutil.copy(BEST, '/content/drive/MyDrive/ped_light_best.pt')
print('Drive 에 저장: MyDrive/ped_light_best.pt')

files.download(BEST)


---
## 다음 단계 (로컬)

1. `best.pt` 를 `models/ped_light.pt` 로 저장
2. `main.py` 에 붙이기 — `SIGNAL_ROI` 크롭 → 이 검출기 → `ped_light` 박스
   → `models/signal_cls.pt` 분류기 → 기존 `SignalVoter` 5프레임 투표

### 주의
- 학습 크롭(`TRAIN_ROI`, 화면 0.30~0.85)과 배포 ROI(`config.SIGNAL_ROI`, 상단 52%)는
  **다르다.** 차량 시점 데이터에서 보행등 중심의 중앙값이 y=0.594 이기 때문이다.
  스케일은 근접하지만(학습 0.53, 배포 0.64) **검증된 것은 아니다.**
- 이 검출기가 배운 것은 **차 안에서 본 보행등**이다. 보행자 눈높이와 각도·거리
  분포가 다르므로, 실사용 전에 직접 촬영한 데이터로 파인튜닝이 필요할 수 있다.
- 녹색 점멸은 이 라벨로 구분되지 않는다. 시간축 추적으로 따로 처리해야 한다.